# Clase 4 · ¿Y qué tan seguro estás de ese número?

**Estadística Descriptiva e Inferencial** · Módulo 2 · Sesión 4 de 14

Del parámetro al estimador: muestreo, distribuciones muestrales, error estándar
e intervalos de confianza.

---

### Lo que este notebook hace distinto

En las clases 1 a 3 calculabas cosas sobre datos que tenías. Aquí vas a hacer algo
que en la vida real es imposible: **vas a conocer la población entera.**

Eso te permite hacer lo único que demuestra si un método funciona: construir 10 000
intervalos y **contar cuántos aciertan**. Cuando en tu trabajo construyas uno solo,
nunca sabrás si acertó. Hoy sí.

| Bloque | Tema | Min |
|---|---|---|
| 1 | Población y muestra: por qué x̄ se mueve | 12 |
| 2 | La distribución muestral y el error estándar | 13 |
| 3 | El IC y su cobertura real | 15 |
| 4 | Proporciones: Wald contra Wilson | 13 |
| 5 | Bootstrap | 12 |

> **SEED = 42.** La misma semilla del deck. El bloque 3 es el obligatorio: si el tiempo
> se acaba, los demás se terminan en casa.

## Celda 0 · Preparación

Ejecuta esta celda primero.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, t, lognorm

SEED = 42
rng = np.random.default_rng(SEED)

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-6):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada (obtenido = None)")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}: obtenido = {float(obtenido):.4f} | "
          f"esperado = {float(esperado):.4f} (tolerancia {tol})")
    if not ok:
        print("      -> revisa este paso antes de continuar.")
    return ok

def check_bool(nombre, condicion, pista=""):
    print(f"{'[OK]' if condicion else '[X ]'} {nombre}")
    if not condicion and pista:
        print(f"      -> {pista}")
    return bool(condicion)

print("Entorno listo | SEED =", SEED)

### Tabla de símbolos → código

| Símbolo | En la población | En tu muestra | En el código |
|---|---|---|---|
| media | μ | x̄ | `MU_POB` / `x_barra` |
| desviación | σ | s | `SD_REAL` / `s` (con `ddof=1`) |
| proporción | p | p̂ | `P_REAL` / `p_hat` |
| error estándar | σ/√n | s/√n | `ee` |

Las variables en MAYÚSCULAS son las de la población: **en la vida real no existen.**
Aquí las tenemos solo porque nosotros fabricamos la población.

---
# Bloque 1 · Población y muestra  ·  12 min

Construimos la población de 40 000 montos de transacción del curso: **lognormal**,
la misma de la Clase 3. Es asimétrica y positiva, como cualquier monto real.

In [ ]:
# ── DEMOSTRACIÓN: la población completa ──────────────────────────────────
N_POB = 40_000
SL = 0.7                                  # sigma en escala logaritmica
ML = np.log(3000) - 0.5 * SL**2           # mu en escala logaritmica

pob = np.random.default_rng(SEED).lognormal(mean=ML, sigma=SL, size=N_POB)

# Los parametros TEORICOS de la lognormal (los que usa el deck)
MU_REAL = np.exp(ML + 0.5 * SL**2)
SD_REAL = MU_REAL * np.sqrt(np.exp(SL**2) - 1)

print(f"Parametros teoricos de la poblacion:")
print(f"  mu     = S/ {MU_REAL:,.2f}")
print(f"  mediana= S/ {np.exp(ML):,.2f}   <- muy por debajo de la media: es asimetrica")
print(f"  sigma  = S/ {SD_REAL:,.2f}")
print()
# Y ahora una distincion que importa y que casi nadie hace explicita:
# MU_REAL es el parametro del MODELO que genero los datos.
# MU_POB es la media de los 40 000 montos que efectivamente salieron.
# No son iguales, porque la poblacion tambien es una realizacion aleatoria.
MU_POB = pob.mean()

print("Y lo que salio en nuestros 40 000 montos concretos:")
print(f"  media  = S/ {MU_POB:,.2f}   <- MU_POB, la media realizada")
print(f"  sigma  = S/ {pob.std(ddof=1):,.2f}")
print()
print(f"MU_POB - MU_REAL = {MU_POB - MU_REAL:+.2f} soles.")
print("Parece irrelevante, y con n=40 lo es (es el 5 % del error estandar).")
print("Con n=2560 el error estandar baja a 47 soles y ese desvio pasa a ser el 36 %.")
print("Cuando muestreamos DE 'pob', el parametro que estamos estimando es MU_POB.")
print("Usaremos MU_POB como objetivo en todas las simulaciones de cobertura.")

plt.hist(pob, bins=120, range=(0, 15000), color=BLUE, alpha=0.8)
plt.axvline(MU_REAL, color=MAG, lw=2.5, label=f"mu = {MU_REAL:,.0f}")
plt.axvline(np.exp(ML), color=GREEN, lw=2.5, ls="--", label=f"mediana = {np.exp(ML):,.0f}")
plt.xlabel("monto de la transaccion (S/)"); plt.ylabel("frecuencia")
plt.title("La poblacion: 40 000 montos", color=NAVY, fontweight="bold")
plt.legend(frameon=False); plt.show()

### Ejercicio 1.1 — Toma tres muestras y compara

Reproduce la tabla de la slide 5: tres muestras de n = 40 de la **misma** población.

**Pista:** `g.choice(pob, size=40, replace=False)` toma una muestra sin reemplazo.
Usa un generador propio `g = np.random.default_rng(SEED)` para que las tres muestras
salgan de una secuencia reproducible.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
g = np.random.default_rng(SEED)
n = 40

filas = []
for k in range(3):
    muestra = g.choice(pob, size=n, replace=False)
    x_barra = muestra.mean()
    s       = muestra.std(ddof=1)
    filas.append({"muestra": k + 1, "x_barra": x_barra, "s": s})

tabla_1 = pd.DataFrame(filas)
print(tabla_1.round(1).to_string(index=False))
print()
print(f"mu de la poblacion (MU_POB) = {MU_POB:,.1f}")
print()
print("Las tres x_barra son distintas y ninguna es mu. Las tres estan bien calculadas.")
print("Eso es exactamente el problema que el resto de la clase resuelve.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check_bool("tomaste 3 muestras", len(tabla_1) == 3),
     check_bool("las tres x̄ son distintas entre sí",
                tabla_1["x_barra"].nunique() == 3,
                "si salen iguales, estás reusando el mismo generador reiniciado"),
     check_bool("ninguna x̄ coincide exactamente con μ",
                all(abs(v - MU_POB) > 1 for v in tabla_1["x_barra"]))]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — ¿Y si la muestra fuera más grande?

Repite lo mismo con n = 40, 200 y 1 000, y mide **cuánto se alejan** las x̄ de μ.

Calcula, para cada n, el error absoluto medio |x̄ − μ| sobre 500 muestras.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
g = np.random.default_rng(SEED)

errores = {}
for n_ in [40, 200, 1000]:
    medias = np.array([g.choice(pob, size=n_, replace=False).mean() for _ in range(500)])
    errores[n_] = np.abs(medias - MU_POB).mean()

for n_, e in errores.items():
    print(f"n = {n_:>5}: error absoluto medio = S/ {e:7.1f}")
print()
print("El error baja con n, pero NO proporcionalmente: pasar de 40 a 1000 es 25 veces")
print("mas trabajo y el error solo baja unas 5 veces. Eso es la raiz cuadrada del bloque 2.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check_bool("el error baja al crecer n",
                errores[40] > errores[200] > errores[1000]),
     check_bool("y baja aprox. con √n, no linealmente (razón 40→1000 entre 3 y 8)",
                3 < errores[40] / errores[1000] < 8,
                f"te dio {errores[40]/errores[1000]:.1f}; √25 = 5 es lo esperado")]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

---
# Bloque 2 · La distribución muestral y el error estándar  ·  13 min

Vas a construir con datos la curva magenta de la slide 8: la distribución de x̄.

Es la distribución que **nunca ves** en la vida real, porque solo tomas una muestra.
Aquí la vemos porque podemos repetir el muestreo 10 000 veces.

### Ejercicio 2.1 — La función que genera la distribución muestral

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def distribucion_muestral(n, reps=10_000, semilla=SEED):
    """Devuelve un array con 'reps' medias de muestras de tamaño n de 'pob'."""
    g = np.random.default_rng(semilla)
    muestras = g.choice(pob, size=(reps, n))   # con reemplazo: es lo estandar aqui
    return muestras.mean(axis=1)

dm40 = distribucion_muestral(40)
print(f"{len(dm40)} medias | primera: {dm40[0]:.1f}")
print(f"centro de la distribucion muestral : {dm40.mean():,.1f}   (MU_POB = {MU_POB:,.1f})")
print(f"ancho  de la distribucion muestral : {dm40.std(ddof=1):,.1f}")

In [ ]:
# ── DEMOSTRACIÓN: las dos distribuciones juntas (la slide 8) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

axes[0].hist(pob, bins=100, range=(0, 15000), color="#9BB4E8", alpha=0.9)
axes[0].axvline(MU_POB, color=NAVY, lw=2)
axes[0].set_title("LA POBLACIÓN: montos individuales", color=NAVY, fontweight="bold")
axes[0].set_xlabel("soles")

axes[1].hist(dm40, bins=70, color=MAG, alpha=0.85)
axes[1].axvline(MU_POB, color=NAVY, lw=2)
axes[1].set_title("LA DISTRIBUCIÓN MUESTRAL de x̄ con n = 40", color=NAVY, fontweight="bold")
axes[1].set_xlabel("promedio de la muestra (soles)")
axes[1].set_xlim(1500, 5000)

plt.tight_layout(); plt.show()

print("Misma unidad, escalas completamente distintas. La de la derecha es angosta y")
print("casi simetrica, aunque la de la izquierda no lo sea. Ese es el TLC trabajando.")

### Ejercicio 2.2 — Comprueba que el ancho es σ/√n

Para n = 10, 40, 160 y 640, compara el ancho **observado** de la distribución muestral
con el que predice la fórmula.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
filas = []
for n_ in [10, 40, 160, 640]:
    dm          = distribucion_muestral(n_)
    ee_empirico = dm.std(ddof=1)
    ee_teorico  = SD_REAL / np.sqrt(n_)
    filas.append({"n": n_, "empirico": ee_empirico, "teorico": ee_teorico,
                  "error_%": 100 * abs(ee_empirico - ee_teorico) / ee_teorico})

tabla_ee = pd.DataFrame(filas)
print(tabla_ee.round(2).to_string(index=False))
print()
print("Cuadruplicar n reduce el EE a la mitad. Mira la columna 'teorico':")
print("  n=10 -> 754,  n=40 -> 377,  n=160 -> 189,  n=640 -> 94.")
print("Cada salto es x4 de esfuerzo por /2 de error. Es la economia del muestreo.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
tabla_ee = pd.DataFrame(filas)
r = []
for _, f in tabla_ee.iterrows():
    err = abs(f["empirico"] - f["teorico"]) / f["teorico"]
    r.append(check_bool(f"n = {int(f['n']):>3}: empírico ≈ σ/√n (dentro del 5 %)",
                        err < 0.05, f"error del {err*100:.1f} %"))
r.append(check("EE teórico con n = 40", tabla_ee.loc[1, "teorico"], 377.19, tol=0.5))
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · El intervalo de confianza y su cobertura real  ·  15 min

**Este es el bloque obligatorio.**

Primero construyes un IC como en la slide 11. Después haces lo que en la vida real
no puedes: construir 10 000 y contar cuántos aciertan.

### Ejercicio 3.1 — Un intervalo, a mano

Escribe la función que devuelve el IC de la media. Tres piezas: x̄, el error estándar
y el multiplicador t.

**Ojo:** el multiplicador es `t.ppf(1 - alpha/2, df=n-1)`, no `1.96`. Con n = 40 vale
2.0227, no 1.96.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def ic_media(muestra, conf=0.95):
    """Devuelve (x_barra, limite_inferior, limite_superior)."""
    n       = len(muestra)
    x_barra = muestra.mean()
    ee      = muestra.std(ddof=1) / np.sqrt(n)
    critico = t.ppf(1 - (1 - conf) / 2, df=n - 1)
    margen  = critico * ee
    return x_barra, x_barra - margen, x_barra + margen

g = np.random.default_rng(SEED)
m1 = g.choice(pob, size=40, replace=False)
xb, lo, hi = ic_media(m1)

print(f"x_barra = S/ {xb:,.1f}")
print(f"IC 95 % = S/ {lo:,.1f}  a  S/ {hi:,.1f}   (ancho: S/ {hi-lo:,.1f})")
print(f"MU_POB  = S/ {MU_POB:,.1f}")
print(f"-> {'contiene mu' if lo <= MU_POB <= hi else 'NO contiene mu'}")
print()
print(f"El multiplicador t con df=39 es {t.ppf(0.975, 39):.4f}, no 1.96.")
print(f"Usar 1.96 daria un margen {100*(1-1.96/t.ppf(0.975,39)):.1f} % mas chico: falsamente preciso.")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
xb, lo, hi = ic_media(m1)
r = [check_bool("el IC está centrado en x̄", abs((lo + hi) / 2 - xb) < 1e-8),
     check_bool("el límite inferior es menor que el superior", lo < hi),
     check("multiplicador t usado (df=39)", (hi - xb) / (m1.std(ddof=1)/np.sqrt(40)),
           2.0227, tol=1e-3)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1 — el error típico es usar 1.96 en lugar de t")

### Ejercicio 3.2 — La cobertura real: 10 000 intervalos

Aquí está el examen del método. Construye 10 000 intervalos al 95 % con n = 40 y
cuenta qué porcentaje contiene μ.

Si el IC funciona como promete, debería salir 95 %. **Presta atención al número que
obtienes.**

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def cobertura(n, reps=10_000, conf=0.95, poblacion=None, mu_real=None, semilla=SEED):
    """% de los 'reps' intervalos que contienen mu_real."""
    poblacion = pob if poblacion is None else poblacion
    mu_real   = MU_POB if mu_real is None else mu_real
    g = np.random.default_rng(semilla)
    muestras = g.choice(poblacion, size=(reps, n))

    x_barra = muestras.mean(axis=1)
    s       = muestras.std(axis=1, ddof=1)
    critico = t.ppf(1 - (1 - conf) / 2, df=n - 1)
    margen  = critico * s / np.sqrt(n)

    dentro = (x_barra - margen <= mu_real) & (mu_real <= x_barra + margen)
    return 100 * dentro.mean()

cob_40 = cobertura(40)
print(f"cobertura REAL con n = 40 : {cob_40:.2f} %")
print(f"cobertura PROMETIDA       : 95.00 %")
print()
print("No es un error tuyo. El IC con t asume que x_barra es normal, y con una poblacion")
print("tan asimetrica y n=40 el TLC todavia no llego. Prometes 95 y entregas menos.")
print()
print("La slide 17 del deck reporta 92.3 % para lo mismo, calculado con otro metodo de")
print("muestreo (i.i.d. de la lognormal teorica, en lugar de remuestrear de esta poblacion")
print("finita). Que dos implementaciones independientes coincidan en la primera decimal")
print("es la mejor senal de que el fenomeno es real y no un artefacto del codigo.")

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
r = [check_bool("calculaste una cobertura entre 85 % y 96 %", 85 < cob_40 < 96,
                "si te da algo fuera de ese rango, revisa la comparación vectorizada"),
     check_bool("y la cobertura real quedó POR DEBAJO del 95 % prometido", cob_40 < 95,
                "con esta población asimétrica y n=40, debería quedar por debajo")]
print()
print(f"Uno de cada {100/(100-cob_40):.0f} intervalos falla, no uno de cada 20.")
print()
print("3.2 OK" if all(r) else "Revisa 3.2")

### Ejercicio 3.3 — ¿Se arregla con más muestra?

Mide la cobertura para n = 10, 40, 160, 640 y 2 560. Y mídela también sobre una
**población normal** con la misma μ y σ, para aislar el efecto de la asimetría.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
pob_normal = np.random.default_rng(SEED + 1).normal(MU_REAL, SD_REAL, N_POB)
MU_NORMAL  = pob_normal.mean()

filas = []
for n_ in [10, 40, 160, 640, 2560]:
    cob_asim = cobertura(n_)
    cob_norm = cobertura(n_, poblacion=pob_normal, mu_real=MU_NORMAL)
    filas.append({"n": n_, "lognormal": cob_asim, "normal": cob_norm})

tabla_cob = pd.DataFrame(filas)
print(tabla_cob.round(2).to_string(index=False))
print()
print("Lectura de la tabla, que es la conclusion del bloque:")
print("  columna 'normal'    -> clava el 95 % desde n=10. El metodo es correcto.")
print("  columna 'lognormal' -> arranca muy por debajo y sube lentamente con n.")
print()
print("Es decir: el problema no es el intervalo, es la ASIMETRIA de la poblacion.")
print("Y si, se arregla con mas n... pero mucho mas n del que la regla de dedo sugiere.")

plt.plot(tabla_cob["n"], tabla_cob["lognormal"], "o-", color=MAG, lw=2.5,
         label="población lognormal (asimétrica)")
plt.plot(tabla_cob["n"], tabla_cob["normal"], "o-", color=BLUE, lw=2.5,
         label="población normal")
plt.axhline(95, color=NAVY, ls="--", lw=1.5, label="95 % prometido")
plt.xscale("log"); plt.xlabel("tamaño de muestra (escala log)")
plt.ylabel("cobertura real (%)")
plt.title("La cobertura real de un IC al 95 %", color=NAVY, fontweight="bold")
plt.legend(frameon=False); plt.show()

In [ ]:
# ── VERIFICACIÓN 3.3 ─────────────────────────────────────────────────────
tabla_cob = pd.DataFrame(filas)
r = [check_bool("con población normal la cobertura ronda el 95 % en todos los n",
                (tabla_cob["normal"] > 93.5).all() and (tabla_cob["normal"] < 96.5).all()),
     check_bool("con población lognormal la cobertura mejora al crecer n",
                tabla_cob["lognormal"].iloc[-1] > tabla_cob["lognormal"].iloc[0]),
     check_bool("y con n = 10 la cobertura lognormal es claramente mala (< 92 %)",
                tabla_cob["lognormal"].iloc[0] < 92)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.3")

---
# Bloque 4 · Proporciones: Wald contra Wilson  ·  13 min

Ahora una tasa de fraude, que es el caso que te toca en el trabajo. Y una tasa **baja**,
que es donde la fórmula del libro se rompe.

$$\text{Wald} = \hat{p} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

`statsmodels` recorta el límite negativo a 0, así que para **ver** el problema hay que
calcular Wald a mano. Eso hacemos.

### Ejercicio 4.1 — Wald a mano, y el límite imposible

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def wald(k, n, conf=0.95):
    """IC de Wald calculado a mano (sin recortar en 0)."""
    p_hat = k / n
    z     = norm.ppf(1 - (1 - conf) / 2)
    h     = z * np.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat, p_hat - h, p_hat + h

for k in [9, 2]:
    p, lo, hi = wald(k, 1200)
    alerta = "  <-- TASA NEGATIVA: IMPOSIBLE" if lo < 0 else ""
    print(f"{k} fraudes de 1200:  p_hat = {p*100:.3f} %   "
          f"IC Wald = [{lo*100:+.3f} %, {hi*100:.3f} %]{alerta}")
print()
print("Y lo que devuelve statsmodels, que recorta el limite:")
from statsmodels.stats.proportion import proportion_confint
for k in [9, 2]:
    lo, hi = proportion_confint(k, 1200, method="normal")
    print(f"  {k} fraudes: [{lo*100:.3f} %, {hi*100:.3f} %]  <- el recorte esconde el problema")
print()
print("La condicion clasica: n*p_hat >= 10. Con 2 fraudes, n*p_hat = 2. Muy por debajo.")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
_, lo9, hi9 = wald(9, 1200)
_, lo2, hi2 = wald(2, 1200)
r = [check("Wald con 9/1200 · límite inferior (%)", lo9 * 100, 0.2618, tol=1e-3),
     check("Wald con 9/1200 · límite superior (%)", hi9 * 100, 1.2382, tol=1e-3),
     check_bool("con 2/1200 el límite inferior de Wald es NEGATIVO", lo2 < 0,
                "si te sale 0, estás usando statsmodels en lugar de la fórmula a mano")]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — Wilson, y la cobertura real de cada uno

Wilson es la alternativa que no se rompe. Su fórmula es más fea pero no hay que
memorizarla: está en `statsmodels`.

Lo que sí vale hacer es **medir la cobertura real de las dos** con una tasa baja.
Ese es el argumento que gana la discusión.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
from statsmodels.stats.proportion import proportion_confint

P_REAL, n_rev, reps = 0.008, 1200, 5000
g = np.random.default_rng(SEED)
exitos = g.binomial(n_rev, P_REAL, size=reps)

# Wald vectorizado a mano
p_hat = exitos / n_rev
z     = norm.ppf(0.975)
h     = z * np.sqrt(p_hat * (1 - p_hat) / n_rev)
cob_wald = 100 * np.mean((p_hat - h <= P_REAL) & (P_REAL <= p_hat + h))

# Wilson con statsmodels (acepta arrays)
lo_w, hi_w = proportion_confint(exitos, n_rev, method="wilson")
cob_wilson = 100 * np.mean((lo_w <= P_REAL) & (P_REAL <= hi_w))

print(f"Tasa real de fraude       : {P_REAL*100:.2f} %   (n*p = {n_rev*P_REAL:.1f})")
print(f"Cobertura real de Wald    : {cob_wald:.2f} %   <- prometia 95 %")
print(f"Cobertura real de Wilson  : {cob_wilson:.2f} %   <- cumple")
print()
print("Con tasas bajas Wald es sistematicamente OPTIMISTA: sus intervalos son demasiado")
print("angostos y fallan mas de lo que dicen. En fraude y AML, donde las tasas son de")
print("0.1 % a 1 %, esto no es un detalle academico: es el rango en el que trabajas siempre.")

# La cobertura de Wald empeora justo cuando la tasa baja
tasas = [0.002, 0.005, 0.01, 0.05, 0.20, 0.50]
comp = []
for pr in tasas:
    e = g.binomial(n_rev, pr, size=2000)
    ph = e / n_rev
    hh = z * np.sqrt(ph * (1 - ph) / n_rev)
    cw = 100 * np.mean((ph - hh <= pr) & (pr <= ph + hh))
    l2, h2 = proportion_confint(e, n_rev, method="wilson")
    cs = 100 * np.mean((l2 <= pr) & (pr <= h2))
    comp.append({"p_real": pr, "n*p": n_rev * pr, "Wald": cw, "Wilson": cs})
print()
print(pd.DataFrame(comp).round(2).to_string(index=False))

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check_bool("Wilson se acerca al 95 % (entre 93 y 97)", 93 <= cob_wilson <= 97),
     check_bool("Wald queda por debajo de Wilson con esta tasa baja",
                cob_wald < cob_wilson,
                "con p=0.008 y n=1200, Wald debería cubrir menos"),
     check_bool("y Wald incumple su promesa de 95 %", cob_wald < 94)]
print()
print("La regla que te llevas: proportion_confint(k, n, method='wilson') por defecto.")
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.2")

---
# Bloque 5 · Bootstrap  ·  12 min

El bloque 3 dejó un problema abierto: con población asimétrica y n moderado, el IC
clásico no cumple. El bootstrap es una de las salidas.

**La idea completa, en tres pasos:**
1. Remuestrea **tu muestra** con reemplazo, del mismo tamaño n.
2. Calcula el estadístico. Repite muchas veces.
3. El IC son los percentiles 2.5 y 97.5 de esos valores.

No asume normalidad y no necesita fórmula, así que sirve para estadísticos que
no tienen una: medianas, percentiles, índices.

### Ejercicio 5.1 — Bootstrap para la mediana

No hay fórmula sencilla para el IC de una mediana. Con bootstrap sale en cinco líneas.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def ic_bootstrap(muestra, estadistico=np.mean, B=5000, conf=0.95, semilla=SEED):
    """IC bootstrap percentil para cualquier estadístico."""
    g = np.random.default_rng(semilla)
    n = len(muestra)
    remuestras = g.choice(muestra, size=(B, n), replace=True)
    valores    = np.apply_along_axis(estadistico, 1, remuestras)
    alpha      = 1 - conf
    return np.percentile(valores, [100 * alpha / 2, 100 * (1 - alpha / 2)])

g = np.random.default_rng(SEED)
m200 = g.choice(pob, size=200, replace=False)

MEDIANA_POB = np.median(pob)
for nombre, est in [("media", np.mean), ("mediana", np.median),
                    ("percentil 95", lambda x: np.percentile(x, 95))]:
    lo, hi = ic_bootstrap(m200, est)
    print(f"{nombre:>13}: estimado = {est(m200):8,.0f}   IC bootstrap = [{lo:8,.0f}, {hi:8,.0f}]")

print()
print(f"Valores reales de la poblacion: media = {MU_POB:,.0f}, mediana = {MEDIANA_POB:,.0f}")
print()
print("Fijate en la mediana: no existe formula cerrada sencilla para su IC,")
print("y con bootstrap salio con el mismo codigo que la media. Eso es todo el atractivo.")

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
lo_m, hi_m = ic_bootstrap(m200, np.median)
lo_x, hi_x = ic_bootstrap(m200, np.mean)
r = [check_bool("el IC bootstrap de la mediana contiene la mediana real",
                lo_m <= np.median(pob) <= hi_m),
     check_bool("el IC bootstrap de la media contiene μ",
                lo_x <= MU_POB <= hi_x),
     check_bool("el IC de la mediana es más angosto que el de la media",
                (hi_m - lo_m) < (hi_x - lo_x),
                "en una lognormal la mediana se estima con más precisión que la media")]
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

### Ejercicio 5.2 — ¿Cubre mejor que el clásico?

La pregunta honesta: en el caso que rompió el bloque 3 (población asimétrica, n = 40),
¿el bootstrap cubre mejor que el IC con t?

Mide las dos coberturas y compara. **Ojo: no des por hecho el resultado.**

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
g = np.random.default_rng(SEED)
reps, n_ = 500, 40

aciertos_t, aciertos_boot = 0, 0
for i in range(reps):
    muestra = g.choice(pob, size=n_, replace=False)

    _, lo_t, hi_t = ic_media(muestra)
    if lo_t <= MU_POB <= hi_t:
        aciertos_t += 1

    lo_b, hi_b = ic_bootstrap(muestra, np.mean, B=1000, semilla=i)
    if lo_b <= MU_POB <= hi_b:
        aciertos_boot += 1

cob_t_40    = 100 * aciertos_t / reps
cob_boot_40 = 100 * aciertos_boot / reps

print(f"cobertura del IC con t        : {cob_t_40:.1f} %")
print(f"cobertura del IC bootstrap    : {cob_boot_40:.1f} %")
print(f"prometido por ambos           : 95.0 %")
print()
print("La conclusion honesta, y no es la que se espera:")
print("el bootstrap percentil TAMPOCO llega al 95 % en este caso. Con una poblacion muy")
print("asimetrica y n=40, ambos metodos subcubren, y el bootstrap simple no es una bala")
print("de plata: hereda la asimetria de la muestra que le diste.")
print()
print("Que si funciona: mas n, o variantes mas sofisticadas (bootstrap-t, BCa), o")
print("cambiar de objetivo y estimar la MEDIANA en lugar de la media, que en una lognormal")
print("es un parametro mas estable y mas informativo para el negocio.")

In [ ]:
# ── VERIFICACIÓN 5.2 ─────────────────────────────────────────────────────
r = [check_bool("mediste ambas coberturas", cob_t_40 is not None and cob_boot_40 is not None),
     check_bool("ambas quedan por debajo del 95 % prometido",
                cob_t_40 < 95 and cob_boot_40 < 95,
                "con esta población y n=40, ninguno de los dos debería llegar a 95"),
     check_bool("ambas están en un rango razonable (85-95 %)",
                85 < cob_t_40 < 95 and 85 < cob_boot_40 < 95)]
print()
print("Bloque 5 COMPLETO — laboratorio terminado" if all(r) else "Revisa 5.2")

---
# Cierre

### Checklist de salida

- [ ] Distingo μ de x̄ y sé por qué solo uno tiene distribución.
- [ ] Sé construir la distribución muestral por simulación.
- [ ] Sé calcular un EE y sé por qué va √n.
- [ ] Construyo un IC con `t.ppf`, no con 1.96.
- [ ] Sé medir la cobertura real de un método en lugar de confiar en su promesa.
- [ ] Uso Wilson para proporciones.
- [ ] Sé qué hace un bootstrap y también qué NO arregla.

### Lo que quedó demostrado con números

| Bloque | Lo que viste |
|---|---|
| 1 | Tres muestras correctas dan tres x̄ distintas y ninguna es μ. |
| 2 | El ancho de la distribución muestral es σ/√n, medido, no afirmado. |
| 3 | Un IC al 95 % sobre población asimétrica con n = 40 cubre menos del 95 %. |
| 4 | Wald incumple su promesa justo en el rango de tasas de fraude. |
| 5 | El bootstrap simple no arregla la subcobertura por asimetría. |

Los bloques 3 y 5 son los que importan: los dos midieron que un método **no cumple lo
que promete**. Esa capacidad — medir la cobertura en lugar de confiar en la fórmula —
es la diferencia entre aplicar estadística y entenderla.

### Reto para la próxima clase

Toma un número que ya hayas reportado en tu trabajo: un promedio, una tasa, un porcentaje.

1. Reconstruye su intervalo de confianza al 95 %.
2. Trae el número que reportaste **y el ancho del intervalo** que lo acompañaba (o que le faltaba).
3. Y la pregunta incómoda: **¿cómo se eligió esa muestra?**

### Clase 5

**Pruebas de hipótesis:** la otra cara de la misma moneda. Verás que un IC que no
contiene un valor es exactamente una prueba que lo rechaza — y entra el p-valor, con
la misma advertencia que le pusimos a Shapiro en la Clase 3.

---
*Estadística Descriptiva e Inferencial · Módulo 2 · Clase 4 · SEED = 42*